# 176 — Embedding Space Analysis

Analyze embedding and unembedding spaces: isotropy measurement,
nearest-neighbor structure, embed-unembed correspondence,
norm distributions, and subspace structure.

## Why This Matters

Embedding space examines the structure of the model's token representation spaces. The embedding and unembedding matrices define the model's interface with language, and their geometry constrains what semantic relationships the model can represent.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.embedding_space_analysis import (
    embedding_isotropy,
    embedding_neighborhood,
    embed_unembed_correspondence,
    embedding_norm_distribution,
    embedding_subspace_structure,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
result = embedding_isotropy(model)
for name in ['embedding', 'unembedding']:
    m = result[name]
    print(f"{name:12s}: isotropy={m['isotropy_ratio']:.4f}  "
          f"eff_dim={m['effective_dimensionality']:.1f}  "
          f"cond={m['condition_number']:.1f}  top_sv={m['top_singular_value']:.4f}")

In [ ]:
result = embedding_neighborhood(model, [0, 25, 50, 75, 99], k=5)
for t in result['per_token']:
    neighbors = ', '.join(f"tok{n['token']}({n['cosine_similarity']:.3f})" for n in t['neighbors'])
    print(f"Token {t['token']:3d} (norm={t['embedding_norm']:.3f}): [{neighbors}]")

In [ ]:
result = embed_unembed_correspondence(model, top_k=5)
print(f"Mean alignment: {result['mean_alignment']:.4f}  Std: {result['std_alignment']:.4f}\n")
print("Most aligned (embed ~= unembed direction):")
for t in result['most_aligned']:
    print(f"  Token {t['token']:3d}: cosine={t['cosine']:.4f}")
print("\nLeast aligned:")
for t in result['least_aligned']:
    print(f"  Token {t['token']:3d}: cosine={t['cosine']:.4f}")

In [ ]:
result = embedding_norm_distribution(model)
print(f"Tokens: {result['n_tokens']}")
print(f"Mean norm: {result['mean_norm']:.4f}  Std: {result['std_norm']:.4f}")
print(f"Range: [{result['min_norm']:.4f}, {result['max_norm']:.4f}]")
print(f"CV (coeff of variation): {result['cv']:.3f}")

In [ ]:
result = embedding_subspace_structure(model, n_components=10)
print(f"Total dims: {result['total_dimensions']}  Dims for 90%: {result['dims_for_90pct']}\n")
for i, (ev, cv) in enumerate(zip(result['explained_variance'], result['cumulative_variance'])):
    bar = '#' * int(ev * 100)
    print(f"  PC{i+1:2d}: {ev:.3f}  cumul={cv:.3f}  {bar}")